# TTT-Dialect: AI Hub 방언 음성 인식

**시작 전 필수:** 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

---
**데이터 업로드 방법 (시작 전 1회만)**
1. 탐색기에서 `C:\Users\kts64\TTT\data\upload_ready\dialect` 폴더 열기
2. 전체 선택 → [drive.google.com](https://drive.google.com) 에서 `TTT-data` 폴더 만들고 업로드
3. 업로드 완료 후 아래 Step 1부터 실행

## Step 1 — Drive 연동

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Drive에서 데이터 폴더 찾기
DATA_DIR = '/content/drive/MyDrive/TTT-data'

if not os.path.exists(DATA_DIR):
    print('TTT-data 폴더를 찾지 못했습니다. Drive 루트 목록:')
    for item in os.listdir('/content/drive/MyDrive'):
        print(f'  {item}')
else:
    wav_cnt  = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.wav'))
    json_cnt = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.json'))
    print(f'Drive 연동 완료!')
    print(f'  wav: {wav_cnt}개 / json: {json_cnt}개')
    print(f'  경로: {DATA_DIR}')

## Step 2 — 패키지 설치 및 코드 클론

In [ ]:
import os

REPO_DIR = '/content/TTT-Dialect'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kts6450/TTT-Dialect.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q openai-whisper transformers librosa soundfile jiwer loguru pyyaml accelerate torchaudio
print('done!')

## Step 3 — GPU 확인 및 모델 로드

In [ ]:
import sys, torch
sys.path.insert(0, '/content/TTT-Dialect')

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from models.base_whisper import KoreanWhisperModel
MODEL_NAME = 'SungBeom/whisper-small-ko'
print(f'모델 로드: {MODEL_NAME}')
model = KoreanWhisperModel(MODEL_NAME)
model.model = model.model.to(DEVICE)
params = model.count_parameters()
print(f'완료! 파라미터: {params["total"]:,}')

## Step 4 — manifest 생성 (AI Hub 방언 데이터)

In [ ]:
import json, os, librosa
from tqdm.notebook import tqdm

SR = 16000
manifest = []

PROVINCE_MAP = {
    'gs': '경상', 'gw': '강원', 'jl': '전라',
    'cc': '충청', 'jj': '제주', 'se': '서울'
}

def parse_label(json_path):
    """방언 데이터 json 파싱 (sentences[].standard 합치기)"""
    with open(json_path, encoding='utf-8') as jf:
        d = json.load(jf)

    trans = d.get('transcription', {})

    # 방법1: 전체 standard 필드
    text = trans.get('standard', '').strip()

    # 방법2: sentences 리스트 합치기
    if not text:
        sentences = trans.get('sentences', [])
        text = ' '.join(
            s.get('standard', '') or s.get('dialect', '')
            for s in sentences
        ).strip()

    # 방언 코드
    speakers = d.get('speaker', [{}])
    province_code = speakers[0].get('residenceProvince', 'gs') if speakers else 'gs'
    dialect = PROVINCE_MAP.get(province_code, province_code)

    # 나이
    birth_year = speakers[0].get('birthYear', 1960) if speakers else 1960
    try:
        age = 2024 - int(birth_year)
    except:
        age = 65

    return text, dialect, age

files = os.listdir(DATA_DIR)
wav_files  = [f for f in files if f.endswith('.wav')]
json_stems = {os.path.splitext(f)[0] for f in files if f.endswith('.json')}

print(f'처리 중: wav {len(wav_files)}개')
added, skip = 0, 0

for wav_fname in tqdm(wav_files):
    try:
        stem = os.path.splitext(wav_fname)[0]
        if stem not in json_stems:
            skip += 1
            continue

        json_path = os.path.join(DATA_DIR, stem + '.json')
        wav_path  = os.path.join(DATA_DIR, wav_fname)

        text, dialect, age = parse_label(json_path)
        if not text:
            skip += 1
            continue

        audio = librosa.load(wav_path, sr=SR)[0]
        duration = len(audio) / SR
        if duration < 0.5 or duration > 60:
            skip += 1
            continue

        manifest.append({
            'audio_path': wav_path,
            'transcript': text,
            'dialect': dialect,
            'speaker_age': age,
            'speaker_id': stem[:12],
            'duration_sec': round(duration, 2),
            'type': 'dialect'
        })
        added += 1
    except Exception as e:
        skip += 1

MANIFEST_PATH = '/content/manifest.jsonl'
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    for m in manifest:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

print(f'added: {added} / skipped: {skip}')
if manifest:
    print(f'예시: "{manifest[0]["transcript"][:50]}"')
    print(f'방언: {manifest[0]["dialect"]} | 나이: {manifest[0]["speaker_age"]}')
else:
    print('0개 — 아래 진단 셀 실행')

## Step 4-진단 (manifest가 0개일 때만 실행)

In [ ]:
import json, os
for f in os.listdir(DATA_DIR):
    if f.endswith('.json'):
        with open(os.path.join(DATA_DIR, f), encoding='utf-8') as jf:
            d = json.load(jf)
        print('keys:', list(d.keys()))
        trans = d.get('transcription', {})
        print('transcription keys:', list(trans.keys()))
        print('standard:', str(trans.get('standard', ''))[:100])
        sents = trans.get('sentences', [])
        if sents:
            print('sentences[0] keys:', list(sents[0].keys()))
            print('sentences[0].standard:', sents[0].get('standard', ''))
        break

## Step 5 — TTT 어댑터 초기화

In [ ]:
from models.ttt_adapter import TTTAdapter

adapter = TTTAdapter(
    base_model=model,
    profile_dir='/content/user_profiles',
    top_k_layers=2,
    lr=1e-4,
    adaptation_steps=30,
)
print('TTT adapter ready!')

## Step 6 — TTT 이전 WER 측정

In [ ]:
import json, librosa, torch
from jiwer import wer
from tqdm.notebook import tqdm

SR = 16000
N_TEST = 50

samples = []
with open(MANIFEST_PATH) as f:
    for line in f:
        samples.append(json.loads(line))

print(f'전체: {len(samples)}개')
if len(samples) == 0:
    raise SystemExit('manifest 비어있음 — Step 4 재실행 필요')

test_samples = samples[:N_TEST]

def infer(sample, mdl):
    audio, _ = librosa.load(sample['audio_path'], sr=SR)
    feat = mdl.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features.to(DEVICE)
    with torch.no_grad():
        result = mdl.transcribe(feat)
    return result[0] if result else ''

print(f'TTT 이전 WER 측정 ({N_TEST}개)...')
refs, hyps = [], []
for s in tqdm(test_samples):
    refs.append(s['transcript'])
    hyps.append(infer(s, model))

wer_before = wer(refs, hyps)
print(f'TTT 이전 WER: {wer_before:.1%}')
print(f'  정답: "{refs[0][:40]}"')
print(f'  인식: "{hyps[0][:40]}"')

## Step 7 — TTT 캘리브레이션 (핵심)

In [ ]:
N_CALIB = 20
calib_samples = samples[N_TEST:N_TEST + N_CALIB]

calib_features, calib_texts = [], []
for s in calib_samples:
    audio, _ = librosa.load(s['audio_path'], sr=SR)
    feat = model.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features[0]
    calib_features.append(feat)
    calib_texts.append(s['transcript'])

print(f'TTT 캘리브레이션: {N_CALIB}개 (약 1~2분)')

profile = adapter.calibrate(
    user_id='aihub_dialect_user',
    audio_features=calib_features,
    transcripts=calib_texts,
    dialect=calib_samples[0].get('dialect', '경상'),
    age=calib_samples[0].get('speaker_age', 65),
)

print(f'TTT 완료!')
print(f'  WER 이전: {profile.wer_before:.1%}')
print(f'  WER 이후: {profile.wer_after:.1%}')
print(f'  개선:     {profile.wer_before - profile.wer_after:.1%}p')

## Step 8 — 결과 시각화 및 Drive 저장

In [ ]:
import matplotlib.pyplot as plt, json, os
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('TTT 적용 전후 성능 비교 (AI Hub 방언 데이터)', fontsize=14, fontweight='bold')

ax = axes[0]
values = [profile.wer_before * 100, profile.wer_after * 100]
bars = ax.bar(['TTT 이전', 'TTT 이후'], values, color=['#FF6B6B', '#4ECDC4'], alpha=0.85, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('WER (%) — 낮을수록 좋음')
ax.set_title('WER 비교')
ax.set_ylim(0, max(values) * 1.3)
ax.spines[['top', 'right']].set_visible(False)

ax2 = axes[1]
if profile.adaptation_history:
    steps = list(range(len(profile.adaptation_history)))
    wers = [h['wer'] * 100 for h in profile.adaptation_history]
    ax2.plot(steps, wers, 'o-', color='#667eea', linewidth=2.5, markersize=8)
    ax2.fill_between(steps, wers, alpha=0.15, color='#667eea')
    ax2.set_xlabel('적응 횟수')
    ax2.set_ylabel('WER (%)')
    ax2.set_title('적응 이력')
    ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

RESULT_DIR = '/content/drive/MyDrive/TTT-results'
os.makedirs(RESULT_DIR, exist_ok=True)
plt.savefig(f'{RESULT_DIR}/wer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

summary = {
    'model': MODEL_NAME,
    'data': 'AI Hub 방언 (경상+강원)',
    'n_test': N_TEST, 'n_calib': N_CALIB,
    'wer_before': profile.wer_before,
    'wer_after': profile.wer_after,
    'improvement_pp': profile.wer_before - profile.wer_after,
    'improvement_pct': (profile.wer_before - profile.wer_after) / max(profile.wer_before, 1e-9) * 100
}
with open(f'{RESULT_DIR}/summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'TTT 이전 WER: {summary["wer_before"]:.1%}')
print(f'TTT 이후 WER: {summary["wer_after"]:.1%}')
print(f'개선: {summary["improvement_pp"]:.1%}p ({summary["improvement_pct"]:.1f}%)')
print(f'결과 저장: {RESULT_DIR}')